In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

# 1. Định nghĩa Masked Convolution
class MaskedConvolution(nn.Module):
    def __init__(self, mask_type, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(MaskedConvolution, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=True)
        
        # Tạo mặt nạ (mask)
        mask = torch.ones_like(self.conv.weight)
        _, _, h, w = mask.shape
        
        mask[:, :, h // 2, w // 2 + (mask_type == 'B'):] = 0
        mask[:, :, h // 2 + 1:, :] = 0
        
        self.register_buffer('mask', mask)

    def forward(self, x):
        self.conv.weight.data *= self.mask
        return self.conv(x)

# 2. Xây dựng khối Residual Block cho PixelCNN
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels // 2, 1),
            nn.ReLU(),
            MaskedConvolution('B', channels // 2, channels // 2, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels // 2, channels, 1),
            nn.ReLU()
        )

    def forward(self, x):
        return x + self.net(x)

# 3. Lắp ghép mô hình PixelCNN
class PixelCNN(nn.Module):
    def __init__(self, num_res_blocks=12, num_channels=128):
        super(PixelCNN, self).__init__()
        
        # Lớp đầu tiên dùng Mask A
        self.input_conv = MaskedConvolution('A', 1, num_channels, 7, padding=3)
        
        # Các khối Residual
        res_blocks = []
        for _ in range(num_res_blocks):
            res_blocks.append(ResidualBlock(num_channels))
        self.res_blocks = nn.Sequential(*res_blocks)
        
        # Lớp đầu ra: Chuyển về 256 mức xám (cho bài toán phân loại pixel)
        self.output_conv = nn.Sequential(
            nn.Conv2d(num_channels, 1024, 1),
            nn.ReLU(),
            nn.Conv2d(1024, 256, 1) # Output 256 để dùng CrossEntropy (mỗi pixel là 1 class)
        )

    def forward(self, x):
        x = self.input_conv(x)
        x = self.res_blocks(x)
        return self.output_conv(x)

# 4. Chuẩn bị dữ liệu MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    # PixelCNN gốc hoạt động trên dữ liệu rời rạc (0-255). 
    # Ở đây ta map về 0 hoặc 1 (Binarized MNIST) để đơn giản hóa như ví dụ Keras
    lambda x: (x > 0.5).float() 
])

train_loader = DataLoader(datasets.MNIST('./data', train=True, download=True, transform=transform), 
                          batch_size=128, shuffle=True)

# 5. Huấn luyện
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PixelCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

def train(epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.to(device)
            target = (data * 255).long().squeeze(1) # Target là giá trị pixel (0-255)
            
            optimizer.zero_grad()
            output = model(data) # Output shape: [B, 256, 28, 28]
            
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# 6. Hàm sinh ảnh (Inference)
def generate_samples():
    model.eval()
    sample = torch.zeros(10, 1, 28, 28).to(device)
    
    with torch.no_grad():
        for i in range(28):
            for j in range(28):
                out = model(sample)
                probs = torch.softmax(out[:, :, i, j], dim=1)
                # Lấy mẫu từ phân phối xác suất được dự đoán
                sample[:, :, i, j] = torch.multinomial(probs, 1).float() / 255.0
                
    # Hiển thị kết quả
    sample = sample.cpu().numpy()
    for i in range(10):
        plt.subplot(2, 5, i+1)
        plt.imshow(sample[i, 0], cmap='gray')
    plt.show()

# Chạy huấn luyện và sinh ảnh
train(10)
generate_samples()

100%|██████████| 9.91M/9.91M [00:01<00:00, 6.69MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 151kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.64MB/s]


In [ ]:
train(10)
generate_samples()

Epoch 1, Loss: 0.1611
Epoch 2, Loss: 0.0912
Epoch 3, Loss: 0.0883
